# Experiment 22 — V3 (KCDP) Prompt Ablation with DeepSeek-V3

Identical to exp21 in every respect except the model: **DeepSeek-V3** (`deepseek-chat`)
replaces Gemini 2.5 Flash. Same 10-student cohort, same 4 ablation conditions,
same human gold standard, same scorer (`utils.metrics`).

Purpose: check whether the ablation pattern observed in exp21 is model-specific
or holds across a different frontier model.

| Condition | rules_mode | kc_mode |
|---|---|---|
| baseline | full (14 rules) | per_problem |
| no_rules | none (0 rules)  | per_problem |
| reduced  | reduced (4 rules) | per_problem |
| no_kc    | full (14 rules) | full_vocab |

Output files: `results/human_validation/ablation/llm_ablation_deepseek_{condition}_{sid}.json`

**Prerequisites:**
- `pip install openai` (uses the OpenAI SDK pointed at DeepSeek's API)
- `DEEPSEEK_API_KEY` set in `.env` — get one free at https://platform.deepseek.com

In [3]:
import json
import os
import sys
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Works on both Windows and WSL2
ROOT = next(
    p for p in [Path("D:/Projects/kintsugi"), Path("/mnt/d/Projects/kintsugi")]
    if p.exists()
)
sys.path.insert(0, str(ROOT))

from lib.v3_prompt import build_v3_prompt
from utils.metrics import evaluate_llm_vs_humans, KC_COLUMNS

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")

# --- Configuration ---
MODEL_ID       = "deepseek-chat"   # DeepSeek-V3
SLEEP_SECONDS  = 2                 # DeepSeek has generous rate limits
TEMPERATURE    = 0.3

STUDENT_IDS = [10155, 9948, 14189, 14352, 14362, 14363, 14374, 14414, 14474, 14499]

INPUT_DIR  = ROOT / "scripts" / "annotation_tool" / "annotation_inputs"
HUMAN_DIR  = ROOT / "dataset" / "Rater_KC_Tags" / "Rated_KC_V3"
OUTPUT_DIR = ROOT / "results" / "human_validation" / "ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_PREFIX = "llm_ablation_deepseek"

CONDITIONS = {
    "baseline": ("full",    "per_problem"),
    "no_rules": ("none",    "per_problem"),
    "reduced":  ("reduced", "per_problem"),
    "no_kc":    ("full",    "full_vocab"),
}
CONDITION_LABELS = {
    "baseline": ("per_problem", "full (14)"),
    "no_rules": ("per_problem", "none (0)"),
    "reduced":  ("per_problem", "reduced (4)"),
    "no_kc":    ("full_vocab",  "full (14)"),
}

VALID_KCS = set(KC_COLUMNS)

pp_df = pd.read_csv(ROOT / "dataset" / "CodeWorkout" / "Problem_Prompts" / "problem_prompts.csv")

def get_required_kcs(problem_id):
    row = pp_df[pp_df["ProblemID"] == problem_id]
    if row.empty:
        return []
    row = row.iloc[0]
    return [kc for kc in KC_COLUMNS if pd.notna(row.get(kc)) and float(row.get(kc)) == 1.0]

def get_problem_info(problem_id):
    row = pp_df[pp_df["ProblemID"] == problem_id]
    if row.empty:
        return None, None
    row = row.iloc[0]
    return row["Requirement"], int(row["AssignmentID"])

print("Project root:", ROOT)
print("Model:", MODEL_ID, "| temp:", TEMPERATURE)
print("Conditions:", list(CONDITIONS))
print("Output dir:", OUTPUT_DIR)

Project root: D:\Projects\kintsugi
Model: deepseek-chat | temp: 0.3
Conditions: ['baseline', 'no_rules', 'reduced', 'no_kc']
Output dir: D:\Projects\kintsugi\results\human_validation\ablation


In [4]:
# --- DeepSeek client via OpenAI SDK ---
if not DEEPSEEK_API_KEY:
    raise ValueError(
        "DEEPSEEK_API_KEY not found.\n"
        "1. Get a free key at https://platform.deepseek.com\n"
        "2. Add  DEEPSEEK_API_KEY='sk-...'  to your .env file"
    )

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)

def call_deepseek(prompt_text):
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt_text}],
        temperature=TEMPERATURE,
    )
    return response.choices[0].message.content or ""

# Quick smoke test
test_reply = call_deepseek("Reply with exactly: {\"ok\": true}")
print("Smoke test:", test_reply[:80])
print("DeepSeek client ready.")

Smoke test: {"ok": true}
DeepSeek client ready.


In [5]:
# --- Load student data + runtime estimate ---
student_data = {}
for sid in STUDENT_IDS:
    with open(INPUT_DIR / f"student_{sid}.json") as f:
        student_data[sid] = json.load(f)

calls_per_condition = sum(
    sum(1 for v in d["submissions"].values() if v["score"] < 1.0)
    for d in student_data.values()
)
print(f"Calls per condition: {calls_per_condition}")
print(f"All 4 conditions: {calls_per_condition * 4} calls")
print(f"~{calls_per_condition * 4 * SLEEP_SECONDS / 60:.0f} min total at {SLEEP_SECONDS}s sleep")

Calls per condition: 188
All 4 conditions: 752 calls
~25 min total at 2s sleep


In [6]:
# --- Parser (same logic as exp21) ---
def parse_llm_response(raw_text):
    cleaned = (raw_text or "").strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned), "ok"
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end   = cleaned.rfind("}")
        if start != -1 and end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1]), "ok_extracted_json"
            except json.JSONDecodeError as e:
                return {"reasoning": raw_text, "knowledge_gaps": []}, f"parse_error: {e}"
        return {"reasoning": raw_text, "knowledge_gaps": []}, "parse_error: no_json_object"

def clean_gaps(parsed):
    gaps = parsed.get("knowledge_gaps", [])
    if isinstance(gaps, str):
        gaps = [gaps]
    if not isinstance(gaps, list):
        return []
    return [g for g in gaps if g in VALID_KCS]

In [7]:
# --- Run loop ---
def run_condition(condition):
    rules_mode, kc_mode = CONDITIONS[condition]
    print(f"\n{'#'*70}\n# CONDITION {condition}  (rules_mode={rules_mode}, kc_mode={kc_mode})\n{'#'*70}")

    for idx, sid in enumerate(STUDENT_IDS):
        out_path = OUTPUT_DIR / f"{FILE_PREFIX}_{condition}_{sid}.json"
        if out_path.exists():
            print(f"[{idx+1}/10] Student {sid} — ALREADY DONE, skipping.")
            continue

        print(f"\n{'='*60}\n[{idx+1}/10] {condition} | Student {sid}\n{'='*60}")
        submissions = student_data[sid]["submissions"]
        annotations, raw_responses, errors = {}, {}, []
        call_count = 0

        for pid_str in sorted(submissions.keys(), key=lambda x: int(x)):
            pid   = int(pid_str)
            sub   = submissions[pid_str]
            score = sub["score"]

            if score >= 1.0:
                annotations[pid_str] = {"gaps": []}
                continue

            requirement, assignment_id = get_problem_info(pid)
            required_kcs = get_required_kcs(pid)
            if requirement is None:
                print(f"  WARNING: Problem {pid} not in problem_prompts.csv, skipping.")
                annotations[pid_str]  = {"gaps": []}
                raw_responses[pid_str] = {"error": "problem_not_found"}
                continue

            prompt = build_v3_prompt(
                problem_id=pid, requirement=requirement, assignment_id=assignment_id,
                required_kcs=required_kcs, student_code=sub["code"], score=score,
                rules_mode=rules_mode, kc_mode=kc_mode,
            )

            call_count += 1
            try:
                t0 = time.time()
                raw_response = call_deepseek(prompt)
                elapsed = time.time() - t0
                parsed, status = parse_llm_response(raw_response)
                gaps = clean_gaps(parsed)
                annotations[pid_str] = {"gaps": gaps}
                raw_responses[pid_str] = {
                    "raw_response":    raw_response,
                    "parsed_response": parsed,
                    "parse_status":    status,
                    "time_sec":        round(elapsed, 3),
                    "score":           score,
                    "assignment_id":   assignment_id,
                    "required_kcs":    required_kcs,
                    "invalid_kcs":     [g for g in parsed.get("knowledge_gaps", [])
                                        if g not in VALID_KCS]
                                       if isinstance(parsed.get("knowledge_gaps", []), list) else [],
                }
                print(f"  P{pid} (score={score:.2f}) -> [{', '.join(gaps) if gaps else '(none)'}] ({elapsed:.1f}s) [{status}]")
            except Exception as e:
                print(f"  ERROR on P{pid}: {e}")
                errors.append(f"API error on P{pid}: {e}")
                annotations[pid_str]  = {"gaps": []}
                raw_responses[pid_str] = {"error": str(e), "score": score,
                                          "assignment_id": assignment_id, "required_kcs": required_kcs}
            time.sleep(SLEEP_SECONDS)

        result = {
            "rater":          f"LLM_DeepSeek_V3_ablation_{condition}",
            "condition":      condition,
            "rules_mode":     rules_mode,
            "kc_mode":        kc_mode,
            "studentId":      str(sid),
            "student_id":     str(sid),
            "model_id":       MODEL_ID,
            "temperature":    TEMPERATURE,
            "exportDate":     datetime.now().isoformat(),
            "total_problems": len(submissions),
            "totalAnnotated": len(annotations),
            "total_calls":    call_count,
            "errors":         errors,
            "annotations":    annotations,
            "raw_responses":  raw_responses,
        }
        with open(out_path, "w") as f:
            json.dump(result, f, indent=2)
        n_gaps = sum(1 for a in annotations.values() if a.get("gaps"))
        print(f"  SAVED {out_path.name} | calls={call_count} with_gaps={n_gaps} errors={len(errors)}")

    print(f"\nCondition {condition} complete.")

## Step 1 — Run the baseline

In [8]:
run_condition("baseline")


######################################################################
# CONDITION baseline  (rules_mode=full, kc_mode=per_problem)
######################################################################

[1/10] baseline | Student 10155
  P3 (score=0.81) -> [LogicAndNotOr] (4.0s) [ok]
  P22 (score=0.36) -> [If/Else, LogicCompareNum, LogicAndNotOr, DefFunction] (4.6s) [ok]
  P24 (score=0.59) -> [LogicCompareNum, LogicAndNotOr] (4.3s) [ok]
  P28 (score=0.20) -> [If/Else, LogicCompareNum, StringFormat, StringConcat, StringIndex] (4.4s) [ok]
  P31 (score=0.27) -> [(none)] (3.3s) [ok]
  P32 (score=0.00) -> [(none)] (3.1s) [ok]
  P33 (score=0.13) -> [(none)] (2.4s) [ok]
  P34 (score=0.50) -> [(none)] (1.9s) [ok]
  P36 (score=0.41) -> [(none)] (1.9s) [ok]
  P37 (score=0.67) -> [StringEqual, LogicAndNotOr] (3.9s) [ok]
  P38 (score=0.73) -> [LogicAndNotOr, StringIndex] (4.7s) [ok]
  P39 (score=0.63) -> [(none)] (1.8s) [ok]
  P40 (score=0.15) -> [(none)] (2.4s) [ok]
  P43 (score=0.62) -> [ArrayI

## Step 2 — Run the remaining three conditions

In [9]:
for cond in ["no_rules", "reduced", "no_kc"]:
    run_condition(cond)


######################################################################
# CONDITION no_rules  (rules_mode=none, kc_mode=per_problem)
######################################################################

[1/10] no_rules | Student 10155
  P3 (score=0.81) -> [LogicAndNotOr] (3.6s) [ok]
  P22 (score=0.36) -> [LogicCompareNum, LogicAndNotOr, DefFunction] (4.1s) [ok]
  P24 (score=0.59) -> [LogicCompareNum, LogicAndNotOr] (3.5s) [ok]
  P28 (score=0.20) -> [LogicCompareNum, StringIndex, StringFormat] (3.3s) [ok]
  P31 (score=0.27) -> [(none)] (2.2s) [ok]
  P32 (score=0.00) -> [(none)] (1.8s) [ok]
  P33 (score=0.13) -> [(none)] (1.7s) [ok]
  P34 (score=0.50) -> [(none)] (1.7s) [ok]
  P36 (score=0.41) -> [(none)] (1.6s) [ok]
  P37 (score=0.67) -> [StringEqual, StringIndex, StringLen, LogicAndNotOr] (3.4s) [ok]
  P38 (score=0.73) -> [StringIndex, LogicAndNotOr] (4.2s) [ok]
  P39 (score=0.63) -> [(none)] (1.8s) [ok]
  P40 (score=0.15) -> [(none)] (3.6s) [ok]
  P43 (score=0.62) -> [If/Else, Logic

## Step 3 — Score and compare with Gemini (exp21)

In [10]:
# --- Scoring helpers ---
def find_one(pattern, directory):
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matched {pattern} in {directory}")
    return matches[-1]

def normalize_gaps(value):
    if isinstance(value, dict):
        gaps = value.get("gaps", [])
    elif isinstance(value, list):
        gaps = value
    else:
        gaps = []
    if not isinstance(gaps, list):
        return set()
    return {g for g in gaps if g in VALID_KCS}

def load_annotation_file(path):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    sid = str(data.get("studentId", data.get("student_id", "unknown")))
    return sid, {f"{sid}_{pid}": normalize_gaps(v) for pid, v in data.get("annotations", {}).items()}

def merge_files(file_map):
    merged = {}
    for expected_sid, path in file_map.items():
        loaded_sid, anns = load_annotation_file(path)
        if loaded_sid != expected_sid:
            raise ValueError(f"Expected {expected_sid}, found {loaded_sid} in {path.name}")
        merged.update(anns)
    return merged

def load_condition(condition, prefix=FILE_PREFIX):
    return merge_files({
        str(sid): OUTPUT_DIR / f"{prefix}_{condition}_{sid}.json"
        for sid in STUDENT_IDS
    })

def parse_rate(condition, prefix=FILE_PREFIX):
    total = ok = 0
    for sid in STUDENT_IDS:
        data = json.loads((OUTPUT_DIR / f"{prefix}_{condition}_{sid}.json").read_text())
        for rec in data.get("raw_responses", {}).values():
            status = rec.get("parse_status")
            if status is None:
                continue
            total += 1
            ok += status.startswith("ok")
    return (ok / total) if total else float("nan")

# Human gold standard
human_a = merge_files({str(sid): find_one(f"kc_annotations_Pranay Ghuge_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})
human_b = merge_files({str(sid): find_one(f"kc_annotations_Arundhati Das_{sid}_*.json", HUMAN_DIR) for sid in STUDENT_IDS})

# Common items defined by DeepSeek baseline ∩ both human raters
common = sorted(
    set(human_a) & set(human_b) & set(load_condition("baseline")),
    key=lambda x: (int(x.split("_")[0]), int(x.split("_")[1]))
)
print(f"Common problem annotations: {len(common)}")

def score_condition(condition, prefix=FILE_PREFIX):
    llm = load_condition(condition, prefix)
    missing = [k for k in common if k not in llm]
    if missing:
        raise ValueError(f"{prefix}/{condition} missing {len(missing)} items")
    avg = evaluate_llm_vs_humans(human_a, human_b, llm, common, KC_COLUMNS)[3]
    return avg["Problem_F1"], avg["Cohen_kappa"], avg["Gwet_AC1"], parse_rate(condition, prefix)

Common problem annotations: 372


In [11]:
# --- Score DeepSeek ablation ---
rows_ds = []
for cond in ["baseline", "no_rules", "reduced", "no_kc"]:
    f1, k, ac1, prate = score_condition(cond)
    kc_inj, rules = CONDITION_LABELS[cond]
    rows_ds.append({"Condition": cond, "KC injection": kc_inj, "Rules": rules,
                    "F1": f1, "κ": k, "AC1": ac1, "Parse %": prate * 100})

print(f"Scored over {len(common)} common annotations.\n")
print("=== DeepSeek-V3 ===")
lines = ["| Condition | KC injection | Rules | F1 | κ | AC1 | Parse % |",
         "|---|---|---|---|---|---|---|"]
for r in rows_ds:
    lines.append(f"| {r['Condition']} | {r['KC injection']} | {r['Rules']} | "
                 f"{r['F1']:.3f} | {r['κ']:.3f} | {r['AC1']:.3f} | {r['Parse %']:.1f}% |")
print("\n".join(lines))

pd.DataFrame(rows_ds)

Scored over 372 common annotations.

=== DeepSeek-V3 ===
| Condition | KC injection | Rules | F1 | κ | AC1 | Parse % |
|---|---|---|---|---|---|---|
| baseline | per_problem | full (14) | 0.825 | 0.535 | 0.953 | 99.5% |
| no_rules | per_problem | none (0) | 0.836 | 0.554 | 0.955 | 98.9% |
| reduced | per_problem | reduced (4) | 0.830 | 0.537 | 0.953 | 99.5% |
| no_kc | full_vocab | full (14) | 0.793 | 0.424 | 0.952 | 98.4% |


,Condition,KC injection,Rules,F1,κ,AC1,Parse %
0,baseline,per_problem,full (14),0.825268,0.534786,0.952875,99.468085
1,no_rules,per_problem,none (0),0.836100,0.554134,0.955400,98.936170
2,reduced,per_problem,reduced (4),0.830252,0.537098,0.953220,99.468085
3,no_kc,full_vocab,full (14),0.792882,0.424487,0.952040,98.404255


In [12]:
# --- Side-by-side comparison: DeepSeek vs Gemini (exp21) ---
# Gemini prefix in exp21 is "llm_ablation" (no model tag)
GEMINI_PREFIX = "llm_ablation"

rows_compare = []
for cond in ["baseline", "no_rules", "reduced", "no_kc"]:
    ds_f1, ds_k, ds_ac1, _ = score_condition(cond, prefix=FILE_PREFIX)
    gm_f1, gm_k, gm_ac1, _ = score_condition(cond, prefix=GEMINI_PREFIX)
    rows_compare.append({
        "Condition": cond,
        "DS F1": ds_f1, "DS κ": ds_k, "DS AC1": ds_ac1,
        "GM F1": gm_f1, "GM κ": gm_k, "GM AC1": gm_ac1,
        "ΔF1 (DS−GM)": ds_f1 - gm_f1,
    })

df_cmp = pd.DataFrame(rows_compare)
print("=== DeepSeek-V3 vs Gemini 2.5 Flash — all conditions ===")
print(df_cmp.to_string(index=False, float_format="{:.3f}".format))

# Save comparison table
md_path = ROOT / "ablation_model_comparison.md"
lines = [
    "# V3 Ablation — DeepSeek-V3 vs Gemini 2.5 Flash\n",
    f"Scored over {len(common)} common problem annotations (10 students).\n",
    "| Condition | DS F1 | DS κ | DS AC1 | GM F1 | GM κ | GM AC1 | ΔF1 |",
    "|---|---|---|---|---|---|---|---|",
]
for r in rows_compare:
    lines.append(
        f"| {r['Condition']} | {r['DS F1']:.3f} | {r['DS κ']:.3f} | {r['DS AC1']:.3f} "
        f"| {r['GM F1']:.3f} | {r['GM κ']:.3f} | {r['GM AC1']:.3f} | {r['ΔF1 (DS−GM)']:+.3f} |"
    )
md_path.write_text("\n".join(lines) + "\n")
print(f"\nWrote {md_path}")

df_cmp

=== DeepSeek-V3 vs Gemini 2.5 Flash — all conditions ===
Condition  DS F1  DS κ  DS AC1  GM F1  GM κ  GM AC1  ΔF1 (DS−GM)
 baseline  0.825 0.535   0.953  0.831 0.548   0.953       -0.006
 no_rules  0.836 0.554   0.955  0.847 0.571   0.952       -0.011
  reduced  0.830 0.537   0.953  0.829 0.533   0.949        0.001
    no_kc  0.793 0.424   0.952  0.820 0.496   0.950       -0.027


UnicodeEncodeError: 'charmap' codec can't encode character '\u03ba' in position 137: character maps to <undefined>